<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.8: A Bus and a UART
**Prev: [Hazards](5.7_hazards.ipynb)**<br>

## Motivation
So far the processor's memory has been ideal: a RAM next to the core that answers in the same
cycle. In a real system, the core reaches memory and devices through a **bus**: it sends a
request, the bus routes it by address, and the answer may take cycles. Devices sit on the same bus
as memory, at their own addresses, and programs control them with ordinary loads and stores:
**memory-mapped I/O**.

This notebook puts the pipelined core of [Module 5.7](5.7_hazards.ipynb) on a bus with the RAM
and a **UART**, the serial port of every microcontroller, and makes it print "Hello, world!".

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# The bus

The core's data port is a `MemPort`: a request (`Decoupled`: address, data, byte strobes) and an
answer (`Valid`: data, or an error). The rules of this bus:

- **Handshake.** A request is taken when `valid` and `ready` are both high in a cycle (`fire`).
  The core then waits for the answer; it sends each request once (`ForwardingCore` remembers that
  it has sent it).
- **One request at a time, in order.** The bus is busy from a request until its answer, so
  answers come back in the order of the requests.
- **Backpressure.** A device answers when it can. Until then the core waits, the whole pipeline
  holding (the `memWait` of 5.7).
- **Errors.** An address that no device holds is answered with an error, which halts the core with
  `BadAddress`, as an address outside the RAM did before.

The bus registers each request, so every load and store now takes a cycle more. Instruction fetch
stays on the RAM's own port, which answers in the same cycle (a fetch through the bus would need a
core that keeps its fetch request when a jump redirects it).

| Address | Device | Register |
|---|---|---|
| `0x8000_0000` - `0x8000_ffff` | RAM (64 KB) | code, data, stack |
| `0x1000_0000` | UART | data: write a byte to send it |
| `0x1000_0004` | UART | status: 1 while a byte is being sent |

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "Bus.scala"))

---
# The UART

A UART sends a byte over one wire, `txd`, one bit at a time, at a rate both sides agree on (the
baud rate; here a bit lasts `divider` clock cycles). The line is 1 when idle. A byte goes as a
*frame* of 10 bits: a **start bit** (0), which tells the receiver a byte is coming, the 8 **data
bits**, least significant first, and a **stop bit** (1), which returns the line to idle.

The UART on the bus has a data register and a status register. A write to the data register while
the previous byte is still being sent is not answered until the transmitter is free: the store
waits, so a program can write bytes one after the other without checking the status.

## Exercise: the transmitter

Complete the two `???` of `riscv/src/main/scala/riscv/UartTx.scala`: the frame to load (stop bit,
data, start bit: the start bit must be in bit 0, as it goes first) and how the frame moves on after
each bit. Restart the kernel and run the notebook again (the cells that build the transmitter or
the system stop with a `NotImplementedError` until then, as do the exercises of 5.2-5.7).

`UartReceiver` is a receiver in software: fed `txd` once per cycle, it returns each byte.

In [ ]:
simulate(new UartTx(divider = 4)) { c =>
  val receiver = new UartReceiver(4)
  val text     = "Hi!"
  val out      = new StringBuilder
  var pending  = text.toList
  var line     = new StringBuilder // txd, one character per bit
  for (cycle <- 0 until 150) {
    val busy = c.io.busy.peek().litToBoolean
    c.io.send.valid.poke((!busy && pending.nonEmpty).B)
    c.io.send.bits.poke(pending.headOption.getOrElse(' ').toInt.U)
    if (!busy && pending.nonEmpty) pending = pending.tail
    val txd = c.io.txd.peek().litToBoolean
    if (cycle % 4 == 0 && cycle < 4 * 12) line += (if (txd) '1' else '0')
    out ++= receiver.sample(txd)
    c.clock.step()
  }
  println(s"txd, one sample per bit: $line")
  println("idle 1, then 'H' (0x48 = 01001000): start 0, data 00010010 (least significant first), stop 1,")
  println("then the start bit of 'i' (each sample of 'H' is taken in the last cycle of its bit)")
  assert(out.toString == text, out.toString)
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val frame = RegInit(1.U(10.W))                    // the bits still to send, the next one in bit 0
  val left  = RegInit(0.U(4.W))                     // how many
  val count = RegInit(0.U(log2Ceil(divider + 1).W)) // cycles left of the current bit
  io.busy := left =/= 0.U
  io.txd  := Mux(io.busy, frame(0), true.B)

  when(!io.busy &amp;&amp; io.send.valid) {
    frame := 1.U(1.W) ## io.send.bits ## 0.U(1.W)
    left  := 10.U
    count := (divider - 1).U
  }.elsewhen(io.busy) {
    when(count === 0.U) {
      frame := frame &gt;&gt; 1
      left  := left - 1.U
      count := (divider - 1).U
    }.otherwise(count := count - 1.U)
  }
</pre></article></div></section></div>

---
# Hello, world

`riscv.SocSystem` is the system on a chip: the forwarding core, the bus, the RAM and the UART.
The program stores each byte of the string to the UART's data register:

In [ ]:
val hello = os.read(os.pwd / "riscv" / "programs" / "src" / "hello.S")
println(hello)

`Soc.run` runs it and decodes what the UART sends. The golden model has a UART too (`UartModel`,
which sends at once; `SocSystem.model` gives it), so `CoSim.compare` can check every instruction the
core retires, the stores to the UART included:

In [ ]:
simulate(new SocSystem) { c =>
  val image = Asm(hello)
  CoSim.compare(c, image).foreach(d => sys.error(d)) // SocSystem's golden model has the UART
  CoSim.load(c, image)
  print(Soc.run(c))
}

---
# What the bus costs

The bus registers each request, so every load and store takes a cycle more than with the RAM next
to the core, and a store to the UART waits for the whole previous frame. In cycles:

In [ ]:
val fact                                          = Asm(os.read(os.pwd / "riscv" / "programs" / "src" / "fact.S"))
def cycles(c: RiscvSystem, image: Seq[Long]): Int = { CoSim.load(c, image); CoSim.timed(c)._2 }
var near                                          = 0
simulate(new ForwardingSystem) { c => near = cycles(c, fact) }
simulate(new SocSystem) { c =>
  println(s"fact: $near cycles with the RAM next to the core, ${cycles(c, fact)} through the bus")
  println(s"hello: ${cycles(c, Asm(hello))} cycles to the ecall (the UART sends 10 bits of 4 cycles per byte)")
}

---
# You're done!

Without CSRs there are no interrupts, so the UART cannot tell the core when it is done: the core
waits on the bus instead. A program could also read the status register in a loop, but its
instructions would then depend on timing (the golden model's UART is never busy), so
`CoSim.compare` could no longer check it instruction by instruction. Waiting on the bus is enough
for a program that only prints.

[Return to the top.](#top)